# Incremental updates to a knowledge graph

Extraction is the expensive part of RAGU — one or two LLM calls per chunk, plus
summarization. Rebuilding a graph because something changed is the easiest way to
waste money, and every write method on `KnowledgeGraph` exists so you do not have
to.

Incremental updates are not only about **adding** knowledge. The same methods cover
three jobs:

| job | why | methods |
|---|---|---|
| **Add** new knowledge | new documents arrived | `upsert_entities`, `upsert_relations` |
| **Forget** stale knowledge | retention policy, a source was retracted, memory budget | `delete_entities`, `delete_relations` |
| **Correct** extraction errors | the model invented a relation, mistyped an entity, wrote a bad description | `update_*`, or `delete_*` + `upsert_*` |

The third is the one people underestimate. An LLM extractor is a lossy process: it
will occasionally label a company as a `PERSON`, hallucinate a relation, or write a
description that contradicts the source. A graph you cannot edit is a graph you
have to rebuild every time you notice one of those.


Swap `StubEmbedder` for `EmbedderOpenAI` and the same code works against a real
graph.

In [ ]:
import tempfile

import numpy as np

from ragu.common.global_parameters import Settings

Settings.language = "english"
Settings.storage_folder = tempfile.mkdtemp(prefix="ragu_incremental_")

from ragu import BuilderArguments, KnowledgeGraph
from ragu.chunker.types import Chunk
from ragu.graph.types import Entity, Relation
from ragu.models.embedder import Embedder

print(f"working in {Settings.storage_folder}")

## A stub embedder

`Embedder` is a two-method interface. Returning deterministic vectors keeps the
notebook offline and reproducible; retrieval quality is irrelevant here, since
nothing below performs a search.

In [ ]:
class StubEmbedder(Embedder):
    """
    Offline embedder returning a deterministic vector per text.

    :param dim: Embedding dimensionality.
    """

    def __init__(self, dim: int = 16) -> None:
        self._dim = dim

    @property
    def dim(self) -> int:
        return self._dim

    def _vector(self, text: str) -> list[float]:
        rng = np.random.default_rng(abs(hash(text)) % (2**32))
        vector = rng.standard_normal(self._dim).astype(np.float32)
        return list(vector / np.linalg.norm(vector))

    async def embed_text(self, text: str, **kwargs) -> list[float]:
        return self._vector(text)

    async def batch_embed_text(self, texts: list[str], desc: str | None = None, **kwargs):
        return [self._vector(text) for text in texts]


# llm=None is legal: without extraction the graph never calls one.
knowledge_graph = KnowledgeGraph(
    llm=None,
    embedder=StubEmbedder(),
    chunker=None,
    builder_settings=BuilderArguments(build_only_vector_context=True),
)

## A helper to watch the graph change

Every step below is judged by these three numbers.

In [ ]:
async def snapshot(label: str) -> dict[str, int]:
    """
    Print and return the current size of every store.

    :param label: Text printed in front of the counts.
    :returns: Counts keyed by store name.
    """
    index = knowledge_graph.index
    counts = {
        "entities": len(await index.graph_backend.get_all_nodes()),
        "relations": len(await index.graph_backend.get_all_edges()),
        "chunks": len(await index.chunks_kv_storage.all_keys()),
    }
    print(f"{label:34} " + "  ".join(f"{k}={v}" for k, v in counts.items()))
    return counts

## Seeding a graph by hand

Normally these records come out of `build_from_docs`. Writing them directly is
faster to read, and it is exactly what you do when repairing a graph anyway.

Chunks come first: entities and relations cite chunk ids as provenance, and
`check_consistency` later verifies those ids exist.

In [ ]:
CHUNKS = [
    Chunk(content="Dennis Ritchie created C at Bell Labs.", chunk_order_idx=0, doc_id="doc-2023"),
    Chunk(content="Ken Thompson and Ritchie built Unix.", chunk_order_idx=1, doc_id="doc-2023"),
    Chunk(content="Bell Labs also produced the transistor.", chunk_order_idx=2, doc_id="doc-2024"),
]
await knowledge_graph.index.upsert_chunks(CHUNKS)

CHUNK_2023_A, CHUNK_2023_B, CHUNK_2024 = (chunk.id for chunk in CHUNKS)

ritchie = Entity(
    entity_name="Dennis Ritchie", entity_type="PERSON",
    description="Creator of the C programming language.",
    source_chunk_id=[CHUNK_2023_A], documents_id=["doc-2023"],
)
bell_labs = Entity(
    entity_name="Bell Labs", entity_type="ORGANIZATION",
    description="Research laboratory in New Jersey.",
    source_chunk_id=[CHUNK_2023_A], documents_id=["doc-2023"],
)
await knowledge_graph.upsert_entities([ritchie, bell_labs])
await knowledge_graph.upsert_relations([
    Relation(
        subject_id=ritchie.id, object_id=bell_labs.id,
        subject_name=ritchie.entity_name, object_name=bell_labs.entity_name,
        relation_type="WORKS_AS", description="Dennis Ritchie worked at Bell Labs.",
        source_chunk_id=[CHUNK_2023_A],
    )
])
baseline = await snapshot("seeded")

## Adding: `upsert` merges, `update` replaces

The two are not synonyms, and picking the wrong one loses data.

| method | on an existing id | use when |
|---|---|---|
| `upsert_entities` | **merges** with what is stored | new source confirms or extends what you know |
| `update_entities` | **replaces** it wholesale | the stored record is wrong |

The merge policy concatenates descriptions — deduplicating repeated sentences — and
takes the union of `source_chunk_id`, `documents_id` and `clusters`.

In [ ]:
await knowledge_graph.upsert_entities([
    Entity(
        entity_name="Bell Labs", entity_type="ORGANIZATION",
        description="Birthplace of Unix and the transistor.",
        source_chunk_id=[CHUNK_2024], documents_id=["doc-2024"],
    )
])

merged = (await knowledge_graph.get_entities([bell_labs.id]))[0]
print(f"description: {merged.description}")
print(f"chunks:      {len(merged.source_chunk_id)}")
print(f"documents:   {merged.documents_id}")

Re-sending text that is already stored does not duplicate it, so an idempotent
ingestion pipeline stays idempotent.

In [ ]:
await knowledge_graph.upsert_entities([
    Entity(
        entity_name="Bell Labs", entity_type="ORGANIZATION",
        description="Birthplace of Unix and the transistor.",
        source_chunk_id=[CHUNK_2024], documents_id=["doc-2024"],
    )
])
print((await knowledge_graph.get_entities([bell_labs.id]))[0].description)

## Correcting extraction errors

### A wrong description — `update_entities`

Suppose the extractor wrote something the source does not support. `update_entities`
overwrites the record, provenance included, so pass the provenance you want to keep.

In [ ]:
await knowledge_graph.update_entities([
    Entity(
        entity_name="Bell Labs", entity_type="ORGANIZATION",
        description="Industrial research laboratory, part of AT&T until 1996.",
        source_chunk_id=[CHUNK_2023_A, CHUNK_2024], documents_id=["doc-2023", "doc-2024"],
    )
])
corrected = (await knowledge_graph.get_entities([bell_labs.id]))[0]
print(f"description: {corrected.description}")
print(f"chunks kept: {len(corrected.source_chunk_id)}")

### A wrong entity type — `delete` then `upsert`

This one has a trap. An entity id is a hash of **name + type**, so changing the type
produces a *different* record. `update_entities` cannot rename a record in place,
and it refuses rather than silently creating a duplicate.

In [ ]:
mistyped = Entity(
    entity_name="Multics", entity_type="PERSON",          # extractor got the type wrong
    description="Time-sharing operating system that preceded Unix.",
    source_chunk_id=[CHUNK_2023_B], documents_id=["doc-2023"],
)
corrected_type = Entity(
    entity_name="Multics", entity_type="PRODUCT",
    description="Time-sharing operating system that preceded Unix.",
    source_chunk_id=[CHUNK_2023_B], documents_id=["doc-2023"],
)

await knowledge_graph.upsert_entities([mistyped])
print(f"PERSON  id: {mistyped.id}")
print(f"PRODUCT id: {corrected_type.id}")
print(f"same record: {mistyped.id == corrected_type.id}\n")

try:
    await knowledge_graph.update_entities([corrected_type])
except ValueError as error:
    print(f"ValueError: {error}")

The fix is explicit: delete the wrong record, insert the right one. Deleting first
also removes any relations attached to the mistyped entity, so re-adding them is
part of the repair.

In [ ]:
await knowledge_graph.delete_entities([mistyped.id])
await knowledge_graph.upsert_entities([corrected_type])

stored = await knowledge_graph.index.graph_backend.get_all_nodes()
print(f"Multics records now: {[(e.entity_name, e.entity_type) for e in stored if e.entity_name == 'Multics']}")

### A hallucinated relation

Relations are addressed by `EdgeSpec` tuples `(subject_id, object_id, relation_id)`.
Passing `None` as the relation id removes every edge between the pair; passing a
concrete id removes exactly one, which is what you want when only one of several
relations is wrong.

In [ ]:
hallucinated = Relation(
    subject_id=ritchie.id, object_id=bell_labs.id,
    subject_name=ritchie.entity_name, object_name=bell_labs.entity_name,
    relation_type="FOUNDED_BY", description="Dennis Ritchie founded Bell Labs.",  # false
    source_chunk_id=[CHUNK_2023_A],
)
await knowledge_graph.upsert_relations([hallucinated])
await snapshot("after the bad relation")

await knowledge_graph.delete_relations([(ritchie.id, bell_labs.id, hallucinated.id)])
await snapshot("after removing just that one")

remaining = (await knowledge_graph.get_relations([(ritchie.id, bell_labs.id, None)]))[0]
print(f"\nsurviving relations: {[r.relation_type for r in remaining]}")

## Forgetting: retention and memory management

A graph that only grows eventually stops being useful — stale facts compete with
current ones during retrieval, and storage is not free. Because every artifact
records the documents it came from, a retention policy is a filter over
`documents_id`.

The rule below is deliberately conservative: drop an entity only when **every**
document backing it is obsolete. An entity also supported by a current document
survives, having lost nothing but a citation.

In [ ]:
OBSOLETE_DOCUMENTS = {"doc-2023"}

all_entities = await knowledge_graph.index.graph_backend.get_all_nodes()
doomed = [
    entity for entity in all_entities
    if entity.documents_id and set(entity.documents_id) <= OBSOLETE_DOCUMENTS
]
survivors = [entity for entity in all_entities if entity not in doomed]

print(f"obsolete documents: {sorted(OBSOLETE_DOCUMENTS)}")
for entity in all_entities:
    verdict = "drop" if entity in doomed else "keep"
    print(f"  {verdict:5} {entity.entity_name:16} documents={entity.documents_id}")

In [ ]:
await knowledge_graph.delete_entities([entity.id for entity in doomed])
after_retention = await snapshot("after applying retention")

print(f"\nentities removed:  {baseline['entities'] + 1 - after_retention['entities']}")
print(f"relations removed: {baseline['relations'] - after_retention['relations']} (cascade)")
print(f"chunks removed:    {baseline['chunks'] - after_retention['chunks']}")

## Verifying the result

Manual writes are not validated on the way in, so audit afterwards.
`check_consistency` looks across stores: relations whose endpoints are missing,
artifacts citing chunks that were never stored, vectors with no matching record.

In [ ]:
print(await knowledge_graph.index.check_consistency())

In [ ]:
await knowledge_graph.upsert_entities([
    Entity(
        entity_name="Ghost Entity", entity_type="PERSON",
        description="Cites a chunk that was never stored.",
        source_chunk_id=["chunk-that-does-not-exist"],
    )
])

report = await knowledge_graph.index.check_consistency()
print(f"consistent: {report.is_consistent}")
for issue in report.errors:
    print(f"  {issue.check}: {issue.message}")

In [ ]:
ghost_id = Entity(entity_name="Ghost Entity", entity_type="PERSON",
                  description="", source_chunk_id=[]).id
await knowledge_graph.delete_entities([ghost_id])
print(await knowledge_graph.index.check_consistency())

## What still needs a model

Everything above ran without one. Two things do not, and both belong at the **end**
of a batch of updates rather than after each one:

| method | needs | why |
|---|---|---|
| `reindex_descriptions()` | LLM | Merged descriptions grow long and repetitive; this re-summarizes the ones past a sentence threshold. Requires `use_llm_summarization=True`. |
| `reindex_community()` | LLM | Communities were detected over the graph as it was. After edits the stored summaries describe a graph that no longer exists — and nothing warns you. |
| `reindex_graph()` | LLM | Both, in that order. |

This is the quiet failure mode of incremental updates: entity and relation search
stays correct after any number of edits, while **global search keeps answering from
stale community summaries** until you reindex.

## Summary

- `upsert_*` merges, `update_*` replaces. Reach for `upsert_*` unless the stored
  record is wrong.
- Changing an entity's name or type changes its id — that is a delete plus an
  insert, not an update. RAGU raises instead of duplicating.
- Deleting an entity takes its relations with it; deleting a relation by explicit
  id leaves its siblings alone.
- Retention is a filter over `documents_id`. Chunks are not reclaimed by entity
  deletion; remove them separately if that is the point.
- Finish a batch with `check_consistency()`, and with `reindex_graph()` if anything
  downstream reads community summaries.
- Pin `Settings.storage_folder`, or every run starts from an empty graph.

In [ ]:
import shutil

await knowledge_graph.index.close()
shutil.rmtree(Settings.storage_folder, ignore_errors=True)
print(f"removed {Settings.storage_folder}")